In [1]:
import glob, json, os, time, random, shutil
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, RandomSampler
from concurrent.futures import ThreadPoolExecutor
try:
    import segmentation_models_pytorch as smp
except ImportError:
    os.system("pip install -q segmentation-models-pytorch==0.6.0")
    import segmentation_models_pytorch as smp
print("smp", smp.__version__, "| torch", torch.__version__, "| cuda", torch.cuda.is_available())

# ---------------- config ----------------
RUN, LABEL, STACK, ARCH = "E4_consensus_unet_pre", "consensus", 3, "unet"   # ARCH: unet | deeplab
PRETRAIN = True      # ImageNet-pretrained ResNet-18 encoder (needs Internet ON)
EPOCHS, PER_EPOCH, BATCH = 100, 16000, 32
SMOKE = False
if SMOKE: EPOCHS, PER_EPOCH = 2, 640
LAB = {"consensus": 0, "worldcover": 1, "dynworld": 2}[LABEL]
KEY = {"consensus": "consensus", "worldcover": "worldcover", "dynworld": "dynamicworld"}[LABEL]
random.seed(0); np.random.seed(0); torch.manual_seed(0)
dev = "cuda"
OUT = "/kaggle/working"

# ---------------- copy data to local disk (fast random reads) ----------------
def find(name): return sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True))
LOCAL = "/tmp/data"; os.makedirs(LOCAL, exist_ok=True)
print("disk /tmp free GB:", round(shutil.disk_usage("/tmp").free / 1e9, 1))
names = ["X_train.npy", "Y_train.npy", "meta_train.npz", "class_counts_train.json", "s1_norm.json",
         "X_train_p2.npy", "Y_train_p2.npy", "meta_train_p2.npz", "class_counts_train_p2.json",
         "X_train_p3.npy", "Y_train_p3.npy", "meta_train_p3.npz", "class_counts_train_p3.json",
         "X_val.npy", "Y_val.npy", "meta_val.npz"]
def cp(n):
    src = find(n)[0]; dst = f"{LOCAL}/{n}"
    if not (os.path.exists(dst) and os.path.getsize(dst) == os.path.getsize(src)): shutil.copyfile(src, dst)
    return n
t0 = time.time()
with ThreadPoolExecutor(4) as ex: list(ex.map(cp, names))
print(f"copied to {LOCAL} in {time.time() - t0:.0f}s")
D = LOCAL

# ---------------- data ----------------
parts = [(f"{D}/X_train.npy", f"{D}/Y_train.npy", np.load(f"{D}/meta_train.npz")["valid"]),
         (f"{D}/X_train_p2.npy", f"{D}/Y_train_p2.npy", np.load(f"{D}/meta_train_p2.npz")["valid"]),
         (f"{D}/X_train_p3.npy", f"{D}/Y_train_p3.npy", np.load(f"{D}/meta_train_p3.npz")["valid"])]
vparts = [(f"{D}/X_val.npy", f"{D}/Y_val.npy", np.load(f"{D}/meta_val.npz")["valid"])]
norm = json.load(open(f"{D}/s1_norm.json"))

class DS(Dataset):
    def __init__(self, parts, train, limit=None):
        self.parts, self.train = parts, train
        self.idx = [(k, int(i)) for k, p in enumerate(parts) for i in np.where(p[2])[0]]
        if limit: self.idx = self.idx[:limit]
        self.X = self.Y = None
        self.mean = np.array(norm["mean"] * 3, np.float32)[:, None, None]
        self.std = np.array(norm["std"] * 3, np.float32)[:, None, None]
        if STACK == 1: self.mean, self.std = self.mean[2:4], self.std[2:4]
    def __len__(self): return len(self.idx)
    def __getitem__(self, j):
        if self.X is None:
            self.X = [np.load(p[0], mmap_mode="r") for p in self.parts]
            self.Y = [np.load(p[1], mmap_mode="r") for p in self.parts]
        k, i = self.idx[j]
        x = np.asarray(self.X[k][i], dtype=np.float32)
        if STACK == 1: x = x[2:4]
        x = (x - self.mean) / self.std
        y = np.asarray(self.Y[k][i])
        yo, yc = y[LAB], y[0]
        if self.train:
            r = random.randint(0, 3)
            if r: x, yo, yc = (np.rot90(a, r, axes=(-2, -1)) for a in (x, yo, yc))
            if random.random() < 0.5: x, yo, yc = (np.flip(a, -1) for a in (x, yo, yc))
        return (torch.from_numpy(np.ascontiguousarray(x)),
                torch.from_numpy(np.ascontiguousarray(yo).astype(np.int64)),
                torch.from_numpy(np.ascontiguousarray(yc).astype(np.int64)))

tr = DS(parts, True); va = DS(vparts, False, limit=320 if SMOKE else None)
print("train samples:", len(tr), "| val samples:", len(va))
g = torch.Generator(); g.manual_seed(0)
tl = DataLoader(tr, batch_size=BATCH, sampler=RandomSampler(tr, replacement=False, num_samples=PER_EPOCH, generator=g),
                num_workers=4, pin_memory=True, drop_last=True, persistent_workers=True, prefetch_factor=4)
vl = DataLoader(va, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)

cnt = np.zeros(9)
for f in (f"{D}/class_counts_train.json", f"{D}/class_counts_train_p2.json", f"{D}/class_counts_train_p3.json"):
    cnt += np.array(json.load(open(f))[KEY])[:9]
w = 1 / np.sqrt(cnt / cnt.sum()); w = w / w.mean()
print("class share %:", (100 * cnt / cnt.sum()).round(1), "\nclass weights:", w.round(2))

# ---------------- model ----------------
Net = {"unet": smp.Unet, "deeplab": smp.DeepLabV3Plus}[ARCH]
try:
    model = Net("resnet18", encoder_weights="imagenet" if PRETRAIN else None,
                in_channels=6 if STACK == 3 else 2, classes=9).to(dev)
except Exception as e:
    raise SystemExit(f"Could not load the ImageNet weights ({e}). Turn Internet ON in Session options and run again.")
print("encoder init:", "ImageNet" if PRETRAIN else "random")
print("arch:", ARCH, "| params (M):", round(sum(p.numel() for p in model.parameters()) / 1e6, 1))
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
scaler = torch.amp.GradScaler("cuda")
ce = nn.CrossEntropyLoss(weight=torch.tensor(w, dtype=torch.float32, device=dev), ignore_index=255)
dice = smp.losses.DiceLoss("multiclass", ignore_index=255)

def metrics(conf):
    conf = conf.double(); d = conf.diag(); row = conf.sum(1); col = conf.sum(0)
    iou = d / (row + col - d); present = row > 0
    return (d.sum() / conf.sum()).item(), iou[present].mean().item(), iou.tolist()

@torch.no_grad()
def evaluate():
    model.eval()
    co = torch.zeros(9, 9, dtype=torch.long); cc = torch.zeros(9, 9, dtype=torch.long)
    ce_sum = 0.0; n = 0
    for x, yo, yc in vl:
        with torch.autocast("cuda", dtype=torch.float16):
            out = model(x.to(dev))
        ce_sum += F.cross_entropy(out.float(), yo.to(dev), ignore_index=255, reduction="sum").item()
        n += int((yo != 255).sum())
        pred = out.argmax(1).cpu()
        for conf, y in ((co, yo), (cc, yc)):
            m = y != 255
            conf += torch.bincount(y[m] * 9 + pred[m], minlength=81).view(9, 9)
    oa, miou, iou = metrics(co); _, miou_c, _ = metrics(cc)
    return ce_sum / n, oa, miou, miou_c, iou

# ---------------- training ----------------
hist = []; best = -1
for ep in range(1, EPOCHS + 1):
    model.train(); t0 = time.time(); ls = 0.0; nb = 0
    for x, yo, _ in tl:
        x = x.to(dev, non_blocking=True); yo = yo.to(dev, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with torch.autocast("cuda", dtype=torch.float16):
            out = model(x)
        loss = ce(out.float(), yo) + dice(out.float(), yo)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        ls += loss.item(); nb += 1
    sched.step()
    t_train = time.time() - t0
    vce, oa, miou, miou_c, iou = evaluate()
    sec = time.time() - t0
    hist.append(dict(epoch=ep, train_loss=ls / nb, val_ce=vce, val_oa=oa, val_miou=miou, val_miou_cons=miou_c, sec=sec))
    json.dump(hist, open(f"{OUT}/{RUN}_history.json", "w"))
    if miou > best:
        best = miou; torch.save(model.state_dict(), f"{OUT}/{RUN}_best.pt")
        json.dump(dict(epoch=ep, miou=miou, oa=oa, iou=iou), open(f"{OUT}/{RUN}_best_info.json", "w"))
    print(f"ep {ep:3d} | loss {ls/nb:.3f} | val CE {vce:.3f} | OA {oa:.3f} | mIoU {miou:.3f} | mIoU(cons) {miou_c:.3f} | {sec:.0f}s ({nb*BATCH/t_train:.0f} samples/s)", flush=True)

torch.save(model.state_dict(), f"{OUT}/{RUN}_last.pt")
info = json.load(open(f"{OUT}/{RUN}_best_info.json"))
print("\nbest epoch:", info["epoch"], "| val mIoU", round(info["miou"], 3), "| val OA", round(info["oa"], 3))
print("per-class val IoU:", dict(zip(["water","trees","grass","flooded","crops","shrub","built","bare","snow"], np.round(info["iou"], 2))))
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
e = [h["epoch"] for h in hist]
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].plot(e, [h["train_loss"] for h in hist]); ax[0].set_title("train loss (CE+Dice)")
ax[1].plot(e, [h["val_ce"] for h in hist]); ax[1].set_title("val CE (unweighted)")
ax[2].plot(e, [h["val_miou"] for h in hist], label="own labels"); ax[2].plot(e, [h["val_miou_cons"] for h in hist], label="consensus"); ax[2].legend(); ax[2].set_title("val mIoU")
for a in ax: a.set_xlabel("epoch")
plt.tight_layout(); plt.savefig(f"{OUT}/{RUN}_curves.png", dpi=120)

smp 0.6.0 | torch 2.11.0+cu130 | cuda True
disk /tmp free GB: 1038.2
copied to /tmp/data in 214s
train samples: 45277 | val samples: 2460
class share %: [ 6.  37.4 15.   1.5  9.5  9.2  6.9 13.4  1.1] 
class weights: [0.94 0.38 0.59 1.86 0.75 0.76 0.88 0.63 2.21]


config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 46.8MB            

model.safetensors: downloading bytes:           |  0.00B            

encoder init: ImageNet
arch: unet | params (M): 14.3
ep   1 | loss 2.042 | val CE 1.352 | OA 0.504 | mIoU 0.279 | mIoU(cons) 0.279 | 156s (129 samples/s)
ep   2 | loss 1.747 | val CE 1.256 | OA 0.588 | mIoU 0.364 | mIoU(cons) 0.364 | 151s (134 samples/s)
ep   3 | loss 1.655 | val CE 1.123 | OA 0.598 | mIoU 0.381 | mIoU(cons) 0.381 | 149s (135 samples/s)
ep   4 | loss 1.564 | val CE 1.044 | OA 0.628 | mIoU 0.423 | mIoU(cons) 0.423 | 150s (135 samples/s)
ep   5 | loss 1.519 | val CE 1.059 | OA 0.620 | mIoU 0.413 | mIoU(cons) 0.413 | 149s (135 samples/s)
ep   6 | loss 1.465 | val CE 0.942 | OA 0.646 | mIoU 0.456 | mIoU(cons) 0.456 | 150s (135 samples/s)
ep   7 | loss 1.440 | val CE 0.988 | OA 0.655 | mIoU 0.446 | mIoU(cons) 0.446 | 150s (135 samples/s)
ep   8 | loss 1.401 | val CE 0.994 | OA 0.640 | mIoU 0.442 | mIoU(cons) 0.442 | 151s (135 samples/s)
ep   9 | loss 1.375 | val CE 1.001 | OA 0.650 | mIoU 0.452 | mIoU(cons) 0.452 | 151s (135 samples/s)
ep  10 | loss 1.322 | val CE 0.947 | O